In [0]:
# Import functions from Multi-Tool HR Agent
# This cell defines the core HR agent logic that routes questions to appropriate tools:
# - Bonus calculation tool (calls UC function)
# - Genie tool (for employee data queries)
from databricks.sdk import WorkspaceClient
import time
import re, os
import mlflow

# Initialize workspace client
w = WorkspaceClient()

# The Genie space ID
SPACE_ID = "01f1a7cce8341affb459c8c51394741b"
VALID_SPACE_ID = "01f1a7cce8341affb459c8c51394741b"

@mlflow.trace
def genie_tool(question, SPACE_ID):
    response = w.api_client.do(
        "POST",
        f"/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        body={"content": question}
    )
    
    conversation_id = response["conversation_id"]
    message_id = response["message_id"]
    
    for _ in range(30):
        time.sleep(2)
        msg = w.api_client.do(
            "GET",
            f"/api/2.0/genie/spaces/{SPACE_ID}/conversations/{conversation_id}/messages/{message_id}"
        )
        
        if msg.get("status") == "COMPLETED":
            for attachment in msg.get("attachments", []):
                if "text" in attachment:
                    return attachment["text"].get("content", "")
    
    return "No Genie response returned."

@mlflow.trace
def hr_agent(question, genie_space_id):
    """Route an HR question to the relevant tool(s) and return a combined response."""
    responses = []
    tools_used = []
    question_lower = question.lower()
    
    # Bonus Tool
    if "bonus" in question_lower:
        salary_match = re.findall(r"\d+", question)
        if salary_match:
            salary = int(salary_match[0])
            bonus = spark.sql(
                f"SELECT hr_catalog.hr_core.calculate_bonus({salary}) AS bonus"
            ).collect()[0]["bonus"]
            tools_used.append("calculate_bonus")
            responses.append(f"Bonus for salary {salary}: {bonus}")
    
    # Genie Tool
    genie_keywords = ["employee", "employees", "leave", "headcount", "training"]
    if any(word in question_lower for word in genie_keywords):
        genie_answer = genie_tool(question, genie_space_id)
        tools_used.append("genie_tool")
        responses.append(genie_answer)
    
    output = ""
    if tools_used:
        output += "Tools Used:\n" + "\n".join(f"- {tool}" for tool in tools_used) + "\n\n"
    output += "\n\n".join(responses)
    
    return output

In [0]:
# Create MLflow PyFunc wrapper for HR Agent
# This cell wraps the HR agent logic in an MLflow model for deployment to Model Serving
# The model handles both bonus calculations (via SQL execution) and Genie queries
import mlflow
import pandas as pd
import logging
import warnings
from mlflow.models import infer_signature
from databricks.sdk import WorkspaceClient
import time
import re

# Suppress non-actionable MLflow warnings on serverless compute.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="An input example was not provided", category=UserWarning)

class HRAgentModel(mlflow.pyfunc.PythonModel):

    def __init__(self, space_id):
        self.space_id = space_id
    
    def predict(self, context, model_input: pd.DataFrame) -> list[str]:
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.sql import StatementState
        import time
        import re
        
        question = model_input["question"][0]
        w = WorkspaceClient()
        
        # Genie Tool
        def genie_tool(question, space_id):
            response = w.api_client.do(
                "POST",
                f"/api/2.0/genie/spaces/{space_id}/start-conversation",
                body={"content": question}
            )
            
            conversation_id = response["conversation_id"]
            message_id = response["message_id"]
            
            for _ in range(30):
                time.sleep(2)
                msg = w.api_client.do(
                    "GET",
                    f"/api/2.0/genie/spaces/{space_id}/conversations/{conversation_id}/messages/{message_id}"
                )
                
                if msg.get("status") == "COMPLETED":
                    for attachment in msg.get("attachments", []):
                        if "text" in attachment:
                            return attachment["text"].get("content", "")
            
            return "No Genie response returned."
        
        # HR Agent Logic
        responses = []
        tools_used = []
        question_lower = question.lower()
        
        # Bonus Tool
        if "bonus" in question_lower:
            salary_match = re.findall(r"\d+", question)
            if salary_match:
                salary = int(salary_match[0])
                # Call UC function via SQL (works in serving endpoints)
                try:
                    sql_result = w.statement_execution.execute_statement(
                        statement=f"SELECT hr_catalog.hr_core.calculate_bonus({salary}) AS bonus",
                        warehouse_id="1cfe7f2931b647ba",
                        wait_timeout="30s"
                    )
                    # Extract result from response (wait for completion)
                    # Check if succeeded (handle both enum and string)
                    state_value = str(sql_result.status.state) if sql_result.status else None
                    if sql_result.status and (sql_result.status.state == StatementState.SUCCEEDED or state_value == "StatementState.SUCCEEDED" or state_value == "SUCCEEDED"):
                        if sql_result.result and sql_result.result.data_array:
                            bonus = sql_result.result.data_array[0][0]
                        else:
                            bonus = "Error: No result returned"
                    else:
                        # Capture detailed error information
                        error_msg = str(sql_result.status.state) if sql_result.status else 'unknown'
                        if sql_result.status and sql_result.status.error:
                            error_msg += f" - {sql_result.status.error.message if hasattr(sql_result.status.error, 'message') else str(sql_result.status.error)}"
                        bonus = f"Error: Statement failed - {error_msg}"
                except Exception as e:
                    bonus = f"Error executing bonus calculation: {str(e)}"
                tools_used.append("calculate_bonus")
                responses.append(f"Bonus for salary {salary}: {bonus}")
        
        # Genie Tool
        genie_keywords = ["employee", "employees", "leave", "headcount", "training"]
        if any(word in question_lower for word in genie_keywords):
            genie_answer = genie_tool(question, self.space_id)
            tools_used.append("genie_tool")
            responses.append(genie_answer)
        
        output = ""
        if tools_used:
            output += "Tools Used:\n" + "\n".join(f"- {tool}" for tool in tools_used) + "\n\n"
        output += "\n\n".join(responses)
        
        return [output]

# Define the model signature explicitly (avoids calling predict during logging).
signature = infer_signature(
    model_input=pd.DataFrame({"question": ["What is the bonus for a salary of 100000?"]}),
    model_output=["10000.0"]
)

with mlflow.start_run(run_name="hr-agent-run"):

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        mlflow.pyfunc.log_model(
            name="hr_agent",
            python_model=HRAgentModel(space_id=VALID_SPACE_ID),
            signature=signature
        )

    run_id = mlflow.active_run().info.run_id

In [0]:
# Register the logged model to Unity Catalog
# This creates a new version of the hr_multi_tool_agent model
model_uri = f"runs:/{run_id}/hr_agent"

mlflow.register_model(
    model_uri=model_uri,
    name="hr_multi_tool_agent"
)

In [0]:
# Grant USE CATALOG permission to the service principal
# This allows the endpoint to access the hr_catalog
# Service principal ID is retrieved from the secret scope

# Get the service principal ID from the secret scope
client_id = dbutils.secrets.get(scope="hr-agent-scope", key="client-id")

# Execute the GRANT statement with the retrieved client ID
spark.sql(f"""
    GRANT USE CATALOG
    ON CATALOG hr_catalog
    TO `{client_id}`
""")

In [0]:
# Grant USE SCHEMA permission to the service principal
# This allows the endpoint to access objects in hr_catalog.hr_core
# Service principal ID is retrieved from the secret scope

# Get the service principal ID from the secret scope
client_id = dbutils.secrets.get(scope="hr-agent-scope", key="client-id")

# Execute the GRANT statement with the retrieved client ID
spark.sql(f"""
    GRANT USE SCHEMA
    ON SCHEMA hr_catalog.hr_core
    TO `{client_id}`
""")


In [0]:
# Grant SELECT permission on training_completions table
# This allows the Genie tool to query employee training data
# Service principal ID is retrieved from the secret scope

# Get the service principal ID from the secret scope
client_id = dbutils.secrets.get(scope="hr-agent-scope", key="client-id")

# Execute the GRANT statement with the retrieved client ID
spark.sql(f"""
    GRANT SELECT
    ON TABLE hr_catalog.hr_core.training_completions
    TO `{client_id}`
""")

In [0]:
# Grant EXECUTE permission on calculate_bonus function
# This allows the endpoint to call the bonus calculation function
# Service principal ID is retrieved from the secret scope

# Get the service principal ID from the secret scope
client_id = dbutils.secrets.get(scope="hr-agent-scope", key="client-id")

# Execute the GRANT statement with the retrieved client ID
spark.sql(f"""
    GRANT EXECUTE
    ON FUNCTION hr_catalog.hr_core.calculate_bonus
    TO `{client_id}`
""")

In [0]:
# Test the deployed endpoint with a bonus calculation question
# This verifies that the bonus tool works correctly
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="hr-agent-endpoint",
    dataframe_records=[
        {
            "question":
            "What is the bonus for a salary of 1000"        }
    ]
)

print(response)

In [0]:
# Test the deployed endpoint with a Genie query
# This verifies that the Genie tool routes employee questions correctly
response = w.serving_endpoints.query(
    name="hr-agent-endpoint",
    dataframe_records=[
        {
            "question":
            "How many active employees do we have?"
        }
    ]
)
print(response)

In [0]:
# Test the deployed endpoint with a question requiring both tools
# This verifies that the agent can use multiple tools in a single query
response = w.serving_endpoints.query(
    name="hr-agent-endpoint",
    dataframe_records=[
        {
            "question":
            "How many active employees do we have and what is the bonus for a salary of 100000?"
        }
    ]
)
print(response)